# 도쿄일렉트론코리아 RAG 교육 - 기초 과정

## 3일차 목차

| 번호 | 주제 | 핵심 내용 |
|---|---|---|
| 1 | 평가 설계와 수동 지표 | 평가셋, Hit@K, Recall@K, Faithfulness |
| 2 | RAGAS 평가 예제 | RAGAS 데이터 구조, 평가 실행, 점수 해석 |
| 3 | Langfuse와 운영 모니터링 | CallbackHandler, trace 확인, TruLens 기본 개념, RAGAS/TruLens 역할 구분 |
| 4 | 청크 기반 RAG의 한계와 Graph RAG | 청크 기반 RAG의 한계, Graph RAG 배경, 엔터티, 관계, 트리플, 온톨로지 |
| 5 | 미니 Graph RAG 예제 | LLM 트리플 추출, 그래프 조회 계획, 답변 생성, 설계 체크리스트 |
| 6 | 미니 RAG 경진대회 안내 | `미니RAG경진대회.ipynb` 진행 방식 확인 |
| 7 | 정리 | 3일 과정 전체 흐름 마무리 |

## 1. 평가 설계와 수동 지표

RAG는 단순히 돌아가는지보다 **잘 찾고, 근거 있게 답하는지**를 확인해야 개선이 가능합니다.

평가 범위는 두 가지로 나눠 봅니다.

1. **Retrieval 평가**: 상위 K개 안에 필요한 근거가 들어왔는가
2. **Generation 평가**: 답변이 문서에 근거했는가

<img src="image/eval.png" width="380">

Retrieval은 **Hit@K / Recall@K 중심**으로 보고,  
Generation은 **Faithfulness(문서 근거를 벗어나지 않았는가) 중심**으로 봅니다.

수동 지표로 검색 실패를 직접 확인해 두면, 프레임워크 기반 평가 결과도 더 쉽게 해석할 수 있습니다.

### 1.1 왜 RAG 평가가 중요한가?

아래는 RAG에서 자주 보는 실패 사례입니다.
검색 문제와 답변 생성 문제가 섞여 있습니다.
평가가 있어야 원인과 개선 여부를 확인할 수 있습니다.

- 벡터 유사도 기반 검색만으로는 중요한 근거 문서가 상위 결과에 자주 들어오지 않음
- 검색은 잘 되었는데 LLM이 문서를 근거로 답하지 않고 자체 추론으로 대답함
- Chunk 크기나 슬라이딩 윈도우 전략을 바꿔도 어떤 설정이 더 효과적인지 판단할 근거가 없음
- Query Rewriting/Expansion을 적용했지만, 오히려 Recall이 떨어져 성능이 악화됨
- BM25·Vector·Hybrid Retrieval 실험을 해도 정량적 지표 없이 “감”으로 방식이 선택됨

따라서 평가의 역할은 문제를 만드는 것이 아니라, 검색 실패·근거 부족·설정 변경 효과를 눈으로 확인하게 해 주는 것입니다.
RAG 파이프라인을 안정적으로 개선하려면 체계적인 평가 기준이 필요합니다.

### 1.2 평가셋이 먼저인 이유

평가의 출발점은 도구가 아니라 **질문·기준 답변·근거**입니다.

- 질문마다 기대 정답이 무엇인지 알 수 있어야 합니다.
- 어떤 페이지나 청크가 근거인지 미리 확인할 수 있어야 합니다.
- 검색 문제인지 생성 문제인지 나눠 볼 수 있어야 합니다.

프레임워크를 쓰더라도, 평가 품질은 결국 평가셋을 얼마나 분명하게 준비했는가에 달려 있습니다.

<img src="image/evaluation_split.svg" width="820">

In [ ]:
import logging  # 라이브러리 내부 로그를 줄입니다.
import warnings  # 불필요한 라이브러리 경고를 줄입니다.
from pathlib import Path  # ChromaDB 저장 폴더가 있는지 확인할 때 씁니다.

warnings.filterwarnings("ignore", message="IProgress not found.*")  # tqdm/ipywidgets 안내 경고를 숨깁니다.
warnings.filterwarnings("ignore", message="Importing .* from 'ragas.metrics' is deprecated.*", category=DeprecationWarning)  # 현재 RAGAS 실행 방식에서 나는 deprecation 경고를 숨깁니다.
warnings.filterwarnings("ignore", message="No supporting evidence provided.*", category=UserWarning)  # TruLens 평가 중 나오는 보조 근거 안내 경고를 숨깁니다.
logging.getLogger("trulens").setLevel(logging.ERROR)  # TruLens 내부 계측 로그와 rating 파싱 경고를 줄입니다.
logging.getLogger("opentelemetry").setLevel(logging.ERROR)  # trace provider 관련 내부 로그를 줄입니다.

from dotenv import load_dotenv  # .env 파일의 환경 변수를 현재 세션에 불러옵니다.
from langchain_chroma import Chroma  # ChromaDB를 LangChain 벡터 저장소로 사용합니다.
from langchain_community.document_loaders import PyPDFLoader  # PDF를 페이지 단위 문서로 읽어옵니다.
from langchain_community.retrievers import BM25Retriever  # 키워드 기반 BM25 검색기를 만듭니다.
from langchain_core.output_parsers import StrOutputParser  # 모델 응답 객체에서 텍스트만 꺼냅니다.
from langchain_core.prompts import PromptTemplate  # 문자열 프롬프트 템플릿을 만드는 도구입니다.
from langchain_core.retrievers import BaseRetriever  # 커스텀 검색기를 LangChain retriever처럼 쓰기 위해 상속합니다.
from langchain_openai import ChatOpenAI, OpenAIEmbeddings  # OpenAI 채팅 모델과 임베딩 모델을 사용합니다.
from langchain_text_splitters import RecursiveCharacterTextSplitter  # 긴 문서를 검색용 청크로 나눕니다.
from ragas import EvaluationDataset, evaluate  # RAGAS 평가 데이터셋과 실행 함수를 불러옵니다.
from ragas.llms import llm_factory  # RAGAS 평가자용 LLM을 최신 래퍼로 준비합니다.
from ragas.metrics import Faithfulness, FactualCorrectness, LLMContextRecall  # RAGAS evaluate와 호환되는 지표입니다.

from pdf_utils import clean_pdf_documents  # PDF 머리말, 쪽번호, 깨진 글리프를 정리하는 공통 유틸입니다.

load_dotenv(".env", override=True)  # 예제 실행에 필요한 API 키를 현재 세션에 반영합니다.

PDF_PATH = "data/금융투자협회_투자길라잡이_2018.pdf"  # 평가에 사용할 원본 PDF입니다.
DB_PATH = "./chroma_index_kofia_guide_cleaned"  # 정제된 문서 벡터를 저장한 ChromaDB 경로입니다.
COLLECTION_NAME = "kofia_guide_cleaned"  # ChromaDB 안에서 사용할 컬렉션 이름입니다.

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")  # 문서와 질문을 같은 벡터 공간으로 바꿉니다.
llm = ChatOpenAI(model="gpt-5-nano", temperature=0)  # 답변 생성과 간단한 평가에 사용할 모델입니다.

def load_split_docs():
    """PDF를 읽고, 정제하고, 평가에 사용할 청크 리스트로 나눕니다."""
    loader = PyPDFLoader(PDF_PATH)  # 원본 PDF를 페이지 단위 Document로 읽습니다.
    docs = clean_pdf_documents(loader.load())  # 출력에 방해되는 PDF 잡음을 줄입니다.
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,  # 청크 하나의 최대 글자 수입니다.
        chunk_overlap=100,  # 앞뒤 청크가 이어지도록 일부 내용을 겹칩니다.
        separators=["\n\n", ".", " ", ""],  # 문단, 문장, 단어 순서로 자연스럽게 나눕니다.
    )
    split_docs = text_splitter.split_documents(docs)  # PDF 페이지를 검색용 청크로 나눕니다.
    for idx, doc in enumerate(split_docs):
        doc.metadata["id"] = idx  # 수동 정답 청크와 검색 결과를 같은 ID로 비교합니다.
    return split_docs

def load_or_create_vectorstore(embeddings, split_docs):
    """기존 ChromaDB가 있으면 불러오고, 없으면 새로 만듭니다."""
    if Path(DB_PATH).exists():
        return Chroma(
            collection_name=COLLECTION_NAME,
            embedding_function=embeddings,
            persist_directory=DB_PATH,
        )

    return Chroma.from_documents(
        documents=split_docs,  # 처음 실행할 때 저장할 검색용 청크들입니다.
        embedding=embeddings,  # 각 청크를 벡터로 바꿀 임베딩 모델입니다.
        persist_directory=DB_PATH,  # 새로 만든 벡터DB를 저장할 폴더입니다.
        collection_name=COLLECTION_NAME,
    )

def reciprocal_rank_fusion(result_lists, k=60):
    """여러 검색기의 결과를 RRF 방식으로 하나의 순위로 합칩니다."""
    scores = {}  # 문서별 누적 점수를 저장합니다.
    for results in result_lists:
        for rank, doc in enumerate(results):
            doc_id = doc.metadata.get("id", doc.page_content)  # 같은 문서인지 비교할 기준입니다.
            score = 1.0 / (k + rank + 1)  # 순위가 높을수록 더 큰 점수를 줍니다.
            if doc_id not in scores:
                scores[doc_id] = {"doc": doc, "score": 0.0}
            scores[doc_id]["score"] += score
    ranked = sorted(scores.values(), key=lambda item: item["score"], reverse=True)
    return [item["doc"] for item in ranked]

class HybridRRFRetriever(BaseRetriever):
    """Dense 검색과 BM25 검색 결과를 RRF로 합치는 간단한 검색기입니다."""
    retrievers: list  # 함께 결합할 검색기 목록입니다.
    k: int = 60  # RRF 점수 계산에 쓰는 완충 상수입니다.

    def _get_relevant_documents(self, query: str, *, run_manager=None):
        results = [retriever.invoke(query) for retriever in self.retrievers]  # 각 검색기의 결과를 따로 구합니다.
        return reciprocal_rank_fusion(results, k=self.k)  # 여러 결과를 하나의 순위로 합칩니다.

    async def _aget_relevant_documents(self, query: str, *, run_manager=None):
        results = [retriever.invoke(query) for retriever in self.retrievers]  # 비동기 호출에서도 같은 병합 로직을 씁니다.
        return reciprocal_rank_fusion(results, k=self.k)

# 3일차 평가와 모니터링 예제에서 공통으로 쓸 검색 자원을 준비합니다.
split_docs = load_split_docs()  # 수동 평가셋과 같은 청킹 기준을 준비합니다.
vectorstore = load_or_create_vectorstore(embeddings, split_docs)  # ChromaDB를 불러오거나 생성합니다.
dense_retriever = vectorstore.as_retriever(search_kwargs={"k": 6})  # 임베딩 기반 검색기입니다.
bm25_retriever = BM25Retriever.from_documents(split_docs)  # 키워드 기반 검색기입니다.
bm25_retriever.k = 6  # Dense 검색기와 후보 수를 맞춥니다.
hybrid_rrf = HybridRRFRetriever(retrievers=[dense_retriever, bm25_retriever], k=60)  # Dense와 BM25를 RRF로 합칩니다.
retriever = hybrid_rrf  # RAGAS, Langfuse, TruLens 예시에서 기본 검색기로 사용합니다.

print("평가 예제 준비 완료")
print("정제된 청크 수:", len(split_docs))


### 1.3 Retrieval Evaluation (Hit@K / Recall@K 중심)

Retrieval 평가는 크게 두 가지를 봅니다.  
하나는 **정답 근거를 최소 1개라도 놓치지 않았는지**, 다른 하나는 **관련 청크를 얼마나 폭넓게 회수했는지**입니다.

| 지표 | 정의 | 언제 보면 좋은가 |
|---|---|---|
| **Hit@K** | 상위 K개 안에 수동으로 지정한 관련 청크가 **하나라도** 들어왔는가 | 정답 근거를 최소 1개라도 찾았는지 빠르게 확인할 때 |
| **Recall@K** | **수동으로 지정한 관련 청크 전체** 중에서, 상위 K개 안에 **몇 개가 들어왔는가** | 필요한 정보를 얼마나 빠짐없이 회수했는지 볼 때 |
| **Precision@K** | **상위 K개 결과** 중에서, 수동으로 지정한 관련 청크가 **몇 개 들어왔는가** | 상위 결과에 잡음이 많은지 볼 때 |
| **MRR / nDCG** | 관련 근거가 얼마나 앞순위에 배치되었는가 | 순위 품질까지 함께 보고 싶을 때 |

여기서 `관련 청크`는 retriever가 가져온 결과가 아니라, 질문별로 **미리 수동 확인해 둔 정답 청크**를 뜻합니다.
즉, Recall은 **정답 전체 기준으로 얼마나 회수했는지**, Precision은 **가져온 결과 기준으로 얼마나 정확한지**를 보는 지표입니다.

기본 평가 지표는 **Hit@K**와 **Recall@K**입니다.
- **Hit@K**: 상위 K개 안에 관련 청크가 하나라도 포함되었는지 평가
- **Recall@K**: 수동으로 지정한 관련 청크가 상위 K개 안에 얼마나 포함되었는지 평가

> 참고: 질문마다 관련 청크를 1개만 두면 Recall@K가 Hit@K와 같아질 수 있습니다.  
가능하면 질문별로 관련 청크를 2개 이상 지정해 두면 Recall@K를 더 분명하게 해석할 수 있습니다.

#### 평가용 데이터 구조

같은 평가셋을 검색 평가와 생성 평가에 함께 사용합니다.  
검색 평가는 관련 청크 ID를 보고, 생성 평가는 기준 답변을 함께 봅니다.

- `query`: 실제 사용자가 할 법한 질문
- `relevant_doc_ids`: Retrieval 평가에서 사용할 수동 확인 관련 청크 ID
- `reference`: 생성 답변 평가와 RAGAS 평가에서 사용할 기준 답변

In [ ]:
test_cases = [
    {
        "query": '왜 금융투자상품 투자는 반드시 본인 명의로 해야 하는가?',
        "relevant_doc_ids": [36],
        "reference": '금융거래는 실지명의로 해야 하며, 차명계좌로 투자하면 명의를 빌려준 사람이 계좌 재산을 인출하거나 담보로 대출받을 수 있어 손실 회복이 어렵기 때문입니다.',
    },
    {
        "query": '투자설명서와 약관을 받아 두고 보관해야 하는 이유는 무엇인가?',
        "relevant_doc_ids": [38],
        "reference": '복잡한 금융투자상품을 정확히 이해하고, 이해되지 않는 부분은 설명을 요청하기 위해 필요합니다. 또한 투자설명서와 약관, 자필 기재 내용은 사후 법적 다툼에서 판단 자료가 될 수 있으므로 투자기간 동안 보관해야 합니다.',
    },
    {
        "query": '가족이나 지인에게 계좌번호와 비밀번호를 알려주면 왜 위험한가?',
        "relevant_doc_ids": [39],
        "reference": '가족이나 친인척처럼 개인 금융정보를 알기 쉬운 사람이 계좌를 유용해 예기치 못한 피해가 발생할 수 있기 때문입니다. 친밀한 관계와 별개로 계좌번호와 비밀번호는 스스로 보호해야 합니다.',
    },
    {
        "query": '전산장애가 있었을 때 매도의사를 입증하려면 어떤 자료를 남겨야 하는가?',
        "relevant_doc_ids": [444, 446],
        "reference": '전산장애 시 고객지원센터, 영업부, 영업점 등에 전화로 대체주문을 시도한 기록처럼 매도의사를 확인할 수 있는 증거를 남겨야 합니다. 매도 주문 사실을 입증하지 못하면 전산장애와 손해 사이의 인과관계가 부정될 수 있습니다.',
    },
    {
        "query": '모바일폰으로 금융거래를 할 때 꼭 챙겨야 할 보안 수칙은 무엇인가?',
        "relevant_doc_ids": [63],
        "reference": '공식 배포처를 확인하고, 모바일폰이나 인터넷에 금융정보를 저장하지 않으며, 비밀번호를 안전하게 관리해야 합니다. 분실 시 금융서비스를 중지하고, SMS나 OTP를 이용하며, 보안 업데이트와 바이러스 검사를 하고, 잠금기능과 잠금비밀번호를 관리해야 합니다.',
    },
    {
        "query": '직원에게 계좌관리를 맡기기로 했다면 계약서에 어떤 내용을 구체적으로 적어야 하는가?',
        "relevant_doc_ids": [101],
        "reference": '투자일임계약서에는 어떤 사항을 직원에게 일임하는지, 직원이 해서는 안 되는 사항은 무엇인지, 투자일임의 범위와 투자대상 금융투자상품 등을 세부적으로 적어야 합니다.',
    },
    {
        "query": '내가 맡긴 적 없는데 직원이 알아서 거래했다면 어떤 유형의 분쟁으로 볼 수 있는가?',
        "relevant_doc_ids": [126],
        "reference": '금융투자회사 임직원이 고객의 개별적 또는 포괄적 위탁 없이 고객의 예탁자산으로 매매거래를 했다면 임의매매 분쟁으로 볼 수 있습니다.',
    },
    {
        "query": '직원에게 투자판단을 맡긴 계약은 법적으로 어떤 성격으로 이해해야 하는가?',
        "relevant_doc_ids": [162],
        "reference": '투자일임계약은 민법상 위임계약의 일종입니다. 투자행위로 발생한 손익은 원칙적으로 위임자에게 귀속되며, 금융투자회사나 임직원이 주의의무를 위반하거나 불법행위를 하지 않았다면 손실 발생만으로 책임을 묻기 어렵습니다.',
    },
    {
        "query": '경험이 부족한 투자자에게 어떤 식으로 권하면 부당권유 문제가 생길 수 있는가?',
        "relevant_doc_ids": [198],
        "reference": '거래 위험성에 대한 올바른 인식을 방해하거나, 고객의 투자상황에 비해 과대한 위험을 수반하는 거래를 적극 권유하면 부당권유 문제가 생길 수 있습니다. 손실보전각서, 근거 없는 단정적 판단, 중요부분 설명 없이 권유하는 행위도 문제가 됩니다.',
    },
    {
        "query": '불완전판매는 왜 부당권유의 한 유형으로 함께 다뤄지는가?',
        "relevant_doc_ids": [125],
        "reference": '금융투자상품의 불완전판매는 보통 부당권유의 한 유형으로 분류되며, 적합성 원칙, 적정성 원칙, 설명의무, 손실보전약정 금지 등을 종합적으로 고려해 민법상 불법행위 여부를 판단하기 때문입니다.',
    },
    {
        "query": '주문이 다르게 체결되거나 누락됐다면 회사는 무엇을 확인했어야 하는가?',
        "relevant_doc_ids": [375, 376],
        "reference": '금융투자회사 직원은 고객 주문을 처리할 때 매도 또는 매수의 구분, 종목, 수량, 가격, 주문유형 등을 구체적으로 확인해야 합니다. 이를 확인하지 않거나 주문입력 실수로 고객 의사와 다르게 매매하면 손해배상책임이 발생할 수 있습니다.',
    },
    {
        "query": '보이스피싱 피해를 입었다면 어떤 순서로 대응해야 하는가?',
        "relevant_doc_ids": [469],
        "reference": '경찰청 112센터에 피해를 신고하고, 경찰청 핫라인으로 사기범 계좌 보유 금융회사에 연결해 즉시 지급정지를 요청합니다. 이후 지급정지 금융회사에 피해구제 신청서를 제출하면 금융회사가 금감원에 채권소멸절차 개시공고를 요청하고, 금감원 공고 후 피해금 환급이 진행됩니다.',
    },
    {
        "query": '유사수신 업체는 어떤 특징을 보이며 왜 위험한가?',
        "relevant_doc_ids": [456],
        "reference": '유사수신 업체는 소개나 권유로만 알 수 있고 회사 실체를 알기 어렵게 보안을 유지하는 경우가 많습니다. 정상 영업으로 고수익을 내기 어려운데도 고금리, 고배당, 연금형 수익보장, 지급보증 등을 약속하거나 다단계 방식으로 자금을 모집해 투자자 피해 위험이 큽니다.',
    },
    {
        "query": '회사와 먼저 직접 분쟁을 풀어보는 방식이 왜 가장 손쉽고 합리적일 수 있는가?',
        "relevant_doc_ids": [108, 109],
        "reference": '금융투자회사나 임직원이 잘못을 인정하고 손해를 배상할 의사가 있다면 당사자 간 합의로 분쟁을 정리할 수 있습니다. 합의 내용의 불공정성이나 이행 문제가 없다면 민법상 화해계약으로 가장 손쉽고 합리적인 해결 수단이 될 수 있습니다.',
    },
    {
        "query": '금융투자회사 직원이 손실을 보전해 주겠다고 약속하면 왜 문제가 되는가?',
        "relevant_doc_ids": [13, 543],
        "reference": '자본시장법 제55조는 수익보장이나 손실보전 약정 같은 손실보전행위를 금지합니다. 이런 약정은 강행법규 위반으로 원칙적으로 무효이므로, 고객은 약정한 내용대로 손실보전이나 수익보장을 요구할 수 없습니다.',
    },
]

print("평가 문항 수:", len(test_cases))  # 평가셋 크기만 확인합니다.


#### Hit@K / Recall@K 계산

같은 평가셋을 Dense, Sparse, Hybrid 검색기에 넣어 비교합니다.  
이렇게 보면 어떤 검색 방식이 필요한 근거를 더 안정적으로 회수하는지 숫자로 확인할 수 있습니다.

In [ ]:
def retrieve_docs(retriever, query, k=5):  # 검색 결과 중 상위 k개만 평가에 사용합니다.
    return retriever.invoke(query)[:k]

# 검색 결과에서 문서 id만 뽑아 비교하기 쉽게 만듭니다.
def retrieved_ids(docs):  # 검색 결과를 ID 목록으로 바꿉니다.
    return [doc.metadata.get("id") for doc in docs]

# 상위 K개 안에 관련 청크가 하나라도 들어왔는지 계산합니다.
def hit_at_k(retrieved_docs, relevant_doc_ids):
    pred_ids = {doc.metadata.get("id") for doc in retrieved_docs}  # 상위 K개의 문서 ID 집합입니다.
    return bool(pred_ids & set(relevant_doc_ids))

# 수동으로 지정한 관련 청크를 상위 K개가 얼마나 회수했는지 계산합니다.
def recall_at_k(retrieved_docs, relevant_doc_ids):
    pred_ids = {doc.metadata.get("id") for doc in retrieved_docs}  # 상위 K개의 문서 ID 집합입니다.
    matched_ids = [doc_id for doc_id in relevant_doc_ids if doc_id in pred_ids]  # 실제로 회수한 관련 청크만 남깁니다.
    return len(matched_ids) / len(relevant_doc_ids), matched_ids

# 질문별 Hit@K 평균을 구합니다.
def mean_hit_at_k(retriever, test_cases, k=5):
    correct = 0  # Hit한 질문 수를 셉니다.
    total = len(test_cases)

    for case in test_cases:
        retrieved_docs = retrieve_docs(retriever, case["query"], k=k)  # 각 질문의 top-k 결과를 가져옵니다.
        if hit_at_k(retrieved_docs, case["relevant_doc_ids"]):
            correct += 1

    return correct / total

# 질문별 Recall@K 평균을 구합니다.
def mean_recall_at_k(retriever, test_cases, k=5):
    scores = []  # 질문별 Recall@K 값을 모읍니다.
    for case in test_cases:
        retrieved_docs = retrieve_docs(retriever, case["query"], k=k)  # 각 질문의 top-k 결과를 가져옵니다.
        recall, _ = recall_at_k(retrieved_docs, case["relevant_doc_ids"])  # 관련 청크를 얼마나 찾았는지 계산합니다.
        scores.append(recall)
    return sum(scores) / len(scores)

In [ ]:
# 검색기별로 Hit@K와 Recall@K를 한 번에 비교합니다.
print("Dense Hit@5:", mean_hit_at_k(dense_retriever, test_cases, k=5))
print("Sparse Hit@5:", mean_hit_at_k(bm25_retriever, test_cases, k=5))
print("Hybrid Hit@5:", mean_hit_at_k(hybrid_rrf, test_cases, k=5))

print()

print("Dense Recall@5:", mean_recall_at_k(dense_retriever, test_cases, k=5))
print("Sparse Recall@5:", mean_recall_at_k(bm25_retriever, test_cases, k=5))
print("Hybrid Recall@5:", mean_recall_at_k(hybrid_rrf, test_cases, k=5))

print()
print("권장 해석: Hit@K는 관련 청크를 하나라도 찾았는지 보여주고,")
print("Recall@K는 수동으로 지정한 관련 청크를 얼마나 빠짐없이 회수했는지 보여줍니다.")

#### Hybrid Retrieval 상세 로그 보기

평균 점수만 보면 어떤 질문에서 실패했는지 알기 어렵습니다.  
질문별로 검색된 ID와 정답 ID를 나란히 보면 개선할 지점이 더 잘 보입니다.

In [ ]:
def debug_retrieval_metrics(retriever, test_cases, k=5):  # 질문별 Hit@K와 Recall@K를 자세히 출력합니다.
    for case in test_cases:
        query = case["query"]
        relevant_ids = case["relevant_doc_ids"]
        retrieved_docs = retrieve_docs(retriever, query, k=k)
        pred_ids = retrieved_ids(retrieved_docs)  # 출력용으로 top-k ID를 정리합니다.
        recall, matched_ids = recall_at_k(retrieved_docs, relevant_ids)  # 관련 청크를 얼마나 찾았는지 계산합니다.
        hit = hit_at_k(retrieved_docs, relevant_ids)  # 하나라도 찾았는지 계산합니다.

        print(f"\n[질문] {query}")
        print(f"- 예측 Top-{k} IDs: {pred_ids}")
        print(f"- 수동 확인한 관련 IDs: {relevant_ids}")
        print(f"- Hit@{k}: {hit}")
        print(f"- Recall@{k}: {recall:.2f}")
        print(f"- 회수된 관련 IDs: {matched_ids}")
        print("-" * 50)

print("=== Hybrid Retrieval 상세 로그 ===")
debug_retrieval_metrics(hybrid_rrf, test_cases, k=5)

**관찰 포인트**

- **Hit@K**가 높을수록 관련 청크를 하나라도 놓치지 않았다고 볼 수 있습니다.
- **Recall@K**가 높을수록 수동으로 지정한 관련 청크를 더 넓게 회수했다고 해석할 수 있습니다.
- 두 지표가 함께 높으면 관련 근거를 안정적으로 회수한 경우에 가깝습니다.
- 두 지표가 어긋나면 청크 경계, 질문 표현, 관련 청크 라벨링 범위를 함께 점검하는 것이 좋습니다.

### 1.4 Generation Evaluation (Faithfulness 중심)

여기서 **Faithfulness**는 답변이 문서 근거를 벗어나지 않았는지,  
즉 **문서에 없는 내용을 덧붙이지 않았는지**를 보는 지표입니다.

검색 결과가 좋아도 생성 답변이 문서 밖의 내용을 섞으면 RAG 품질은 낮아집니다.  
따라서 생성 평가는 답변 문장 자체보다 **검색된 청크에 근거했는가**를 중심으로 봅니다.

| 평가 항목 | 무엇을 보는가 |
|---|---|
| **Faithfulness** | 답변이 검색된 청크 안의 근거를 벗어나지 않았는가 |
| **Factual Correctness** | 기준 답안과 비교했을 때 사실적으로 맞는가 |
| **Answer Relevance** | 질문에 필요한 방향으로 답했는가 |

이 예제에서는 가장 기본적인 안전장치인 **Faithfulness**만 직접 확인합니다.

#### LLM을 활용한 Faithfulness 평가

`LLM-as-a-Judge`는 다른 LLM에게 답변 품질을 평가하게 하는 방식입니다.
사람이 모든 답변을 직접 읽기 어려울 때, **1차 자동 점검기**로 활용하기 좋습니다.

In [ ]:
judge_prompt = """
다음은 RAG 모델이 생성한 답변과, 참조 문서(Context)입니다.
답변이 문서에 근거했는지 0~1 점수로 평가하세요.

- 1: 문서에서 근거를 명확히 가지고 있음
- 0: 문서에 없는 내용을 생성함 (hallucination)
- 0.5: 부분적으로만 근거가 있음

[질문]
{query}

[답변]
{answer}

[문맥]
{context}

0, 0.5, 1 중 하나의 숫자만 출력하세요.
"""

judge_template = PromptTemplate.from_template(judge_prompt)  # 평가용 문자열을 재사용 가능한 템플릿으로 바꿉니다.

def evaluate_answer(query, answer, context):  # 답변 하나의 faithfulness 점수를 계산합니다.
    judge_llm = ChatOpenAI(model="gpt-5-nano", temperature=0)  # 평가도 같은 소형 모델로 진행합니다.
    judge_chain = judge_template | judge_llm | StrOutputParser()  # 평가 프롬프트와 평가 모델을 하나의 체인으로 묶습니다.
    return judge_chain.invoke({"query": query, "answer": answer, "context": context})  # 답변의 faithfulness 점수를 계산합니다.

In [ ]:
def format_context_for_evaluation(docs):  # 평가에 사용할 검색된 청크를 하나의 문자열로 합칩니다.
    chunks = []  # 문서별 텍스트를 순서대로 모읍니다.
    for doc in docs:
        page = doc.metadata.get("page")  # 문서 페이지가 있으면 함께 남깁니다.
        page_label = f"p.{page + 1}" if isinstance(page, int) else "page unknown"
        chunks.append(f"[출처: {page_label}]\n{doc.page_content}")
    return "\n\n".join(chunks)

answer_prompt = PromptTemplate.from_template("""
당신은 문서 기반 QA 시스템입니다.
아래 문맥(Context)에 기반해서만 답변하세요.
문맥에 없는 내용은 '문서에 해당 정보가 없습니다.'라고 답하세요.
답변은 2문장 이내로 간결하게 작성하세요.

[질문]
{query}

[문맥]
{context}

답변:
""")

query = "금융투자상품 투자는 누구의 명의로 해야 하는가?"  # Faithfulness 평가에 사용할 질문입니다.

docs = hybrid_rrf.invoke(query)  # Hybrid 검색으로 관련 문서를 가져옵니다.
context = format_context_for_evaluation(docs)  # 검색 결과를 평가용 문맥으로 합칩니다.

answer = llm.invoke(answer_prompt.format(query=query, context=context)).content  # 문맥 기반 답변을 생성합니다.
score = evaluate_answer(query, answer, context)  # 생성 답변이 문서에 근거했는지 채점합니다.

print("=== 답변 ===")
print(answer)

print("\n=== Faithfulness Score ===")
print(score)

### 1.5 RAGAS란?

RAGAS는 RAG 시스템을 평가하기 위한 프레임워크입니다.  
질문, 검색된 문맥, 생성 답변, 기준 답변을 한 행으로 묶고, 여러 질문을 같은 방식으로 반복 평가할 수 있게 해 줍니다.

RAGAS가 제공하는 지표는 원리상 직접 구현할 수도 있습니다.  
앞에서 만든 `Hit@K`, `Recall@K`, `Faithfulness`처럼 평가셋, 평가 프롬프트, 채점 코드, 집계 코드를 직접 만들면 비슷한 값을 계산할 수 있습니다.

RAGAS를 쓰는 이유는 직접 구현과 다른 종류의 평가를 하기 위해서가 아닙니다.  
자주 쓰는 RAG 평가 방식을 정해진 데이터 구조와 지표로 묶어, 여러 문항과 여러 실험을 더 자동화되고 균일한 방식으로 비교하기 위해 사용합니다.

RAGAS 지표마다 필요한 입력은 조금씩 다릅니다.

| 지표 | 무엇을 보는가 | 필요한 입력 |
|---|---|---|
| `context_recall` | 기준 답을 만드는 데 필요한 근거가 검색된 청크 안에 들어왔는가 | 질문, 검색된 청크, 기준 답변 |
| `faithfulness` | 생성 답변이 검색된 청크에 근거하고 있는가 | 질문, 검색된 청크, 생성 답변 |
| `factual_correctness` | 생성 답변이 기준 답안과 사실적으로 맞는가 | 생성 답변, 기준 답변 |

`test_cases`에는 질문, 관련 청크 ID, 기준 답변이 함께 들어 있습니다.  
따라서 직접 만든 수동 지표와 RAGAS 평가를 같은 평가셋으로 이어서 볼 수 있습니다.

차이는 평가 가능 여부가 아니라, 직접 구현해서 볼 것인지 프레임워크에 맡겨 반복 실행할 것인지에 있습니다.

| 확인할 점 | 직접 구현 예 | RAGAS 지표 | RAGAS를 쓰면 편한 점 |
|---|---|---|---|
| 검색된 청크가 기준 답변을 뒷받침하는가 | `Hit@K`, `Recall@K`, 직접 만든 청크 평가 코드 | `context_recall` | 청크와 기준 답변 기반 평가를 같은 형식으로 반복 실행하기 쉽습니다. |
| 답변이 검색된 청크에 근거하는가 | 직접 만든 LLM-as-a-Judge Faithfulness | `faithfulness` | 평가 프롬프트와 점수 계산을 매번 새로 작성하지 않아도 됩니다. |
| 답변이 기준 답변과 사실적으로 맞는가 | 수동 채점, 기준 답변 비교 코드 | `factual_correctness` | 문항별 점수와 평균 점수를 한 번에 비교하기 쉽습니다. |

그래서 작은 실험에서는 직접 만든 지표로 원리를 확인하고, 문항 수가 늘어나거나 여러 설정을 비교할 때는 RAGAS로 평가를 자동화하는 흐름이 자연스럽습니다.

## 2. RAGAS 입력 구조 확인

RAGAS를 쓰려면 평가 데이터를 정해진 형태로 맞춰야 합니다.  
핵심은 질문, 검색된 청크, 생성 답변, 기준 답변을 같은 행에 묶는 것입니다.

RAGAS 평가에 들어가는 핵심 입력은 네 가지입니다.

| 필드 | 의미 |
|---|---|
| `user_input` | 사용자가 넣은 질문 |
| `retrieved_contexts` | 검색기로 가져온 문맥 리스트 |
| `response` | 생성 모델이 만든 최종 답변 |
| `reference` | 사람이 준비한 기준 답변 |

이 네 필드가 있으면 `context_recall`, `faithfulness`, `factual_correctness` 같은 지표를 볼 수 있습니다.
다만 `faithfulness`는 생성 답변을 여러 문장으로 쪼갠 뒤 검색된 청크와 비교하므로 청크가 길거나 많을수록 토큰을 많이 씁니다.
`factual_correctness`도 답변과 기준 답변을 여러 사실문으로 쪼개 비교하기 때문에 답변이 길면 무거워질 수 있습니다.

In [ ]:
def extract_context_texts(docs):  # RAGAS 평가용 문맥 리스트를 만듭니다.
    return [doc.page_content for doc in docs]

def extract_pages(docs):  # 디버깅용 페이지 번호를 따로 모읍니다.
    pages = []  # 검색된 페이지를 순서대로 저장합니다.
    for doc in docs:
        page = doc.metadata.get("page")
        pages.append(page + 1 if isinstance(page, int) else None)
    return pages

def build_ragas_rows_from_test_cases(cases, retriever, k=2):  # test_cases를 RAGAS 입력 행으로 바꿉니다.
    rows = []  # 질문별 평가 데이터를 모읍니다.
    for case in cases:
        query = case["query"]  # 현재 평가 질문입니다.
        docs = retriever.invoke(query)[:k]  # 현재 질문의 상위 문서를 검색합니다.
        context = format_context_for_evaluation(docs)  # 생성 답변에 넣을 문맥 문자열입니다.
        response = llm.invoke(answer_prompt.format(query=query, context=context)).content  # 같은 문맥으로 답변을 생성합니다.
        rows.append(
            {
                "user_input": query,  # RAGAS의 사용자 질문 필드입니다.
                "retrieved_contexts": extract_context_texts(docs),  # 검색된 청크 리스트입니다.
                "response": response,  # 모델이 생성한 답변입니다.
                "reference": case["reference"],  # 사람이 준비한 기준 답변입니다.
                "relevant_doc_ids": case["relevant_doc_ids"],  # 수동 검색 평가에 쓰는 정답 청크 ID입니다.
                "retrieved_doc_ids": retrieved_ids(docs),  # 실제 검색된 청크 ID입니다.
                "retrieved_pages": extract_pages(docs),  # 실제 검색된 페이지입니다.
            }
        )
    return rows

ragas_rows = build_ragas_rows_from_test_cases(test_cases[:5], retriever, k=2)  # 앞 5문항을 RAGAS 입력 형태로 바꿉니다.

ragas_rows[0]


`relevant_doc_ids`, `retrieved_doc_ids`, `retrieved_pages`는 RAGAS 필수 입력은 아니지만, 실패 원인을 볼 때 유용합니다.

- `relevant_doc_ids`가 검색 결과에 없다면 Retrieval 문제일 가능성이 큽니다.
- 검색된 청크는 맞게 들어왔는데 답변이 흔들리면 Generation 문제일 가능성이 큽니다.
- 두 경우가 섞여 있으면 청킹, 검색 방식, 프롬프트를 함께 점검해야 합니다.

RAGAS 평가는 위 구조에서 `user_input`, `retrieved_contexts`, `response`, `reference`를 사용해 실행합니다.

### 2.1 RAGAS 평가 실행

위에서 만든 `ragas_rows`는 그대로 RAGAS 평가에 사용할 수 있습니다.  
`relevant_doc_ids`, `retrieved_doc_ids`, `retrieved_pages`는 사람이 실패 원인을 보기 위한 보조 정보이므로, RAGAS에는 핵심 네 필드만 넘깁니다.

예제에서는 API 시간과 비용을 줄이기 위해 앞 2개 문항과 가장 가벼운 지표만 평가합니다.
전체 문항을 평가하려면 `ragas_eval_sample = ragas_rows`로 바꾸면 됩니다.
평가자 모델은 reasoning 토큰을 쓰지 않는 `gpt-4.1-mini`를 사용합니다.
`Faithfulness`와 `FactualCorrectness`는 출력 JSON이 길어질 수 있어 기본 실행에서는 제외합니다.

In [ ]:
from openai import OpenAI  # RAGAS llm_factory에 넘길 OpenAI client입니다.
from ragas.run_config import RunConfig  # RAGAS 평가 실행 시간과 동시 요청 수를 조절합니다.

evaluator = llm_factory(
    "gpt-4.1-mini",
    client=OpenAI(timeout=120.0, max_retries=3),  # API 응답이 늦을 때를 대비해 timeout과 재시도를 늘립니다.
)  # RAGAS 평가자 모델을 준비합니다.

ragas_run_config = RunConfig(
    max_workers=1,  # 평가 문항/metric에 대한 LLM judge 호출을 동시에 여러 개 보내지 않고 1개씩 순차 실행합니다. 요청 폭주, rate limit, timeout 가능성을 줄여 실습 환경에서 더 안정적으로 돌리기 위한 설정입니다.
    timeout=300,  # 각 평가 작업이 오래 걸릴 수 있어 최대 대기 시간을 300초로 넉넉하게 둡니다.
    max_retries=2,  # 일시적인 API 오류나 timeout이 발생하면 최대 2번까지 다시 시도합니다.
)
ragas_eval_sample = ragas_rows[:2]  # 빠른 확인을 위해 앞 2개 문항만 평가합니다.

ragas_evaluation_rows = [
    {
        "user_input": row["user_input"],  # 사용자 질문입니다.
        "retrieved_contexts": row["retrieved_contexts"],  # 검색된 문맥 리스트입니다.
        "response": row["response"],  # 생성 모델이 만든 답변입니다.
        "reference": row["reference"],  # 사람이 준비한 기준 답변입니다.
    }
    for row in ragas_eval_sample
]

ragas_dataset = EvaluationDataset.from_list(ragas_evaluation_rows)  # RAGAS가 읽을 평가셋으로 바꿉니다.
ragas_result = evaluate(
    dataset=ragas_dataset,  # RAGAS 형식으로 바꾼 평가 데이터입니다.
    metrics=[LLMContextRecall()],  # 필요한 청크를 회수했는지만 가볍게 평가합니다.
    llm=evaluator,  # 위에서 준비한 평가자 모델로 지표를 계산합니다.
    run_config=ragas_run_config,  # timeout과 동시 요청 수 설정을 적용합니다.
    show_progress=False,  # 평가 진행률 표시줄을 숨깁니다.
)
result_df = ragas_result.to_pandas()  # 문항별 평가 결과를 표 형태로 꺼냅니다.
result_df


In [ ]:
metric_columns = [
    column
    for column in result_df.columns
    if column == "context_recall" or column.startswith("context_recall")
]  # RAGAS 버전에 따라 지표 컬럼명이 조금 달라질 수 있어 실제 컬럼에서 찾습니다.

if not metric_columns:
    print("context_recall 지표 컬럼을 찾지 못했습니다.")
    print("실제 결과 컬럼:", list(result_df.columns))
else:
    print("RAGAS 요약 점수")
    for column in metric_columns:
        print(f"{column}: {result_df[column].mean():.4f}")  # 지표별 평균 점수를 출력합니다.
    print(f"현재 지표 평균: {result_df[metric_columns].mean().mean() * 100:.2f} / 100")  # 지표 평균을 100점으로 환산합니다.


문항 수를 늘리면 RAGAS가 지표별로 여러 번 LLM을 호출합니다.
전체 평가가 필요하면 위 셀의 `ragas_eval_sample` 범위를 조정합니다.

결과를 볼 때는 현재 실행한 지표가 무엇인지 먼저 확인합니다.

- 기본 실행에서는 비용과 시간을 줄이기 위해 `context_recall`만 봅니다.
- `context_recall`이 낮으면 필요한 근거를 검색하지 못했을 가능성이 큽니다.
- 답변이 검색된 청크 밖의 내용을 섞는지 보려면 `Faithfulness()`를 추가합니다.
- 기준 답변과 사실적으로 맞는지 보려면 `FactualCorrectness()`를 추가합니다.

`Hit@K`, `Recall@K`, RAGAS 점수를 함께 보면 검색 문제와 생성 문제를 더 잘 나눌 수 있습니다.

## 3. Langfuse와 운영 모니터링

RAGAS는 여러 질문을 모아 품질을 점수로 비교하는 데 좋습니다.
하지만 점수만 보면 **왜 실패했는지**는 바로 보이지 않습니다.

RAG 요청은 보통 아래 단계를 거칩니다.

```text
질문 -> 검색 -> 문맥 구성 -> 프롬프트 -> LLM 호출 -> 최종 답변
```

Langfuse는 이 과정을 하나의 trace로 묶어 보여줍니다.
답변이 틀렸을 때 검색이 문제인지, 문맥 구성이 문제인지, 생성 단계가 문제인지 더 빨리 좁혀볼 수 있습니다.

### 3.1 Langfuse 설정

Langfuse Cloud를 사용할 때는 `.env`에 아래 값을 넣습니다.

```bash
LANGFUSE_PUBLIC_KEY=...
LANGFUSE_SECRET_KEY=...
LANGFUSE_HOST=https://cloud.langfuse.com
```

자체 호스팅 환경이라면 `LANGFUSE_HOST`를 사내 Langfuse 주소로 바꾸면 됩니다.
`.env` 파일은 API Key와 마찬가지로 외부 저장소에 올리지 않도록 주의합니다.

<img src="image/langfuse_project.png" width="650">

<img src="image/langfuse_api_key.png" width="650">


In [ ]:
from dotenv import load_dotenv  # .env 파일의 LANGFUSE_* 값을 현재 세션에 불러옵니다.
from langfuse.langchain import CallbackHandler  # LangChain 실행 기록을 Langfuse trace로 보냅니다.

load_dotenv(".env", override=True)  # Langfuse 설정을 현재 노트북 세션에 반영합니다.

langfuse_handler = CallbackHandler()  # 이후 invoke에서 사용할 추적 핸들러입니다.
print("Langfuse CallbackHandler 준비 완료")

### 3.2 CallbackHandler로 RAG 요청 추적하기

Langfuse의 `CallbackHandler`를 `invoke()`의 `config`에 넣으면 LangChain 실행 흐름이 trace로 남습니다.
아래 예시는 3일차에서 불러온 `retriever`를 사용해 간단한 RAG 체인을 만들고, 한 번의 질문을 Langfuse에 기록합니다.


In [ ]:
from langchain_core.prompts import ChatPromptTemplate  # 질문과 문맥을 넣을 프롬프트 틀입니다.
from langchain_core.runnables import RunnableLambda, RunnablePassthrough  # retriever와 함수를 LCEL로 연결합니다.

trace_llm = ChatOpenAI(model="gpt-5-nano", temperature=0)  # trace 예제용 생성 모델입니다.

trace_prompt = ChatPromptTemplate.from_template("""
당신은 문서 기반 QA 시스템입니다.
아래 문맥에 있는 내용만 근거로 답변하세요.
문맥에 없으면 "문서에 해당 정보가 없습니다."라고 답하세요.

[질문]
{question}

[문맥]
{context}

답변:
""")

def format_context_for_trace(docs):  # 검색 결과를 trace에서 읽기 쉬운 문맥 문자열로 바꿉니다.
    parts = []  # 문서별 문맥을 순서대로 모읍니다.
    for doc in docs:
        page = doc.metadata.get("page")
        page_label = f"p.{page + 1}" if isinstance(page, int) else "page unknown"
        parts.append(f"[{page_label}]\n{doc.page_content}")
    return "\n\n".join(parts)

trace_rag_chain = (
    {
        "question": RunnablePassthrough(),
        "context": retriever | RunnableLambda(format_context_for_trace),
    }
    | trace_prompt
    | trace_llm
    | StrOutputParser()
)

trace_question = "보이스피싱 피해금 환급절차에서 공고 기간은 얼마인가요?"
trace_config = {
    "callbacks": [langfuse_handler],
    "metadata": {"section": "day3-langfuse"},
    "tags": ["day3", "rag-trace"],
}

trace_answer = trace_rag_chain.invoke(trace_question, config=trace_config)
print(trace_answer)

실행 후 Langfuse 화면에서는 입력 질문, 검색 결과, 프롬프트, 모델 응답 흐름을 순서대로 확인할 수 있습니다.

<img src="image/langfuse_tracing.png" width="800">

같은 방식으로 `batch()`, `stream()`, `ainvoke()`에도 callback config를 넘기면 실행 방식이 달라도 일관되게 trace를 남길 수 있습니다.


### 3.3 TruLens 기본 개념

TruLens는 RAG 앱을 실행할 때마다 입력, 검색된 청크, 최종 답변을 기록하고 그 기록에 **feedback 함수**를 붙여 품질을 확인하는 도구입니다.  
RAGAS가 준비된 평가셋을 한 번에 평가하는 방식이라면, TruLens는 앱 실행 기록마다 feedback 점수를 붙이는 방식에 가깝습니다.

정답 라벨이 많이 준비되어 있지 않아도, 실제 체인 실행 결과를 대상으로 LLM 기반 feedback을 붙여볼 수 있다는 점이 특징입니다.  
예를 들어 사용자의 질문, 검색된 문맥, 최종 답변이 하나의 record로 남고, 그 record에 아래와 같은 평가가 붙습니다.

RAG에서 자주 보는 TruLens feedback은 세 가지입니다.

| feedback | 확인하는 질문 |
|---|---|
| `Context Relevance` | 검색된 청크가 질문과 관련 있는가 |
| `Groundedness` | 답변이 검색된 청크에 근거하고 있는가 |
| `Answer Relevance` | 최종 답변이 질문에 잘 답하고 있는가 |

실행 구조는 아래처럼 이해하면 됩니다.

```text
RAG 체인 실행
  -> 질문, 검색된 청크, 답변 기록
  -> Context Relevance / Groundedness / Answer Relevance 계산
  -> 요청 단위 record에 feedback 결과 저장
```

아래 셀은 대시보드 대신 현재 요청 1건의 feedback 점수만 표로 확인합니다.
한 번 실행한 결과만 보면 RAGAS와 비슷한 점수표처럼 보이지만, 차이는 이 점수가 앱 실행 record에 붙는다는 점입니다.
코드의 `Metric`은 위 표에 있는 feedback 하나를 실행 가능한 지표로 만든 객체입니다.
최근 TruLens는 검색된 청크를 고를 때 `select_context()` 대신 `on_context()`를 쓰는 방식이 더 안정적입니다.


In [ ]:
import numpy as np  # 청크별 평가 점수를 평균 낼 때 사용합니다.

from langchain_core.output_parsers import StrOutputParser  # LLM 응답에서 문자열만 꺼냅니다.
from langchain_core.runnables import RunnableLambda  # 일반 함수를 체인 단계로 감쌉니다.
from langchain_core.runnables import RunnablePassthrough  # 입력값을 그대로 다음 단계로 넘깁니다.

from trulens.apps.langchain import TruChain  # LangChain 체인을 TruLens 기록 대상으로 감쌉니다.
from trulens.core import Metric  # TruLens feedback 지표를 정의합니다.
from trulens.providers.openai import OpenAI  # OpenAI 모델을 TruLens 평가자로 사용합니다.


trulens_provider = OpenAI(
    model_engine="gpt-5-mini"  # RAG 결과를 평가할 LLM 모델입니다.
)


trulens_retriever = vectorstore.as_retriever(
    search_kwargs={"k": 2}  # 질문마다 관련 청크 2개를 검색합니다.
)


trulens_rag_chain = (  # TruLens로 실행 기록을 남길 RAG 체인입니다.
    {
        "question": RunnablePassthrough(),  # 입력 질문을 그대로 question에 넣습니다.
        "context": trulens_retriever  # 입력 질문으로 관련 문서를 검색합니다.
        | RunnableLambda(format_context_for_trace),  # 검색 문서를 문자열 문맥으로 바꿉니다.
    }
    | trace_prompt  # question과 context로 프롬프트를 만듭니다.
    | trace_llm  # 프롬프트를 LLM에 전달해 답변을 생성합니다.
    | StrOutputParser()  # LLM 응답 객체에서 텍스트만 추출합니다.
)


f_groundedness = (  # 답변이 검색 문맥에 근거했는지 평가합니다.
    Metric(
        implementation=trulens_provider.groundedness_measure_with_cot_reasons,
        name="Groundedness",  # TruLens 화면에 표시될 지표 이름입니다.
    )
    .on_context(collect_list=True)  # 검색된 문맥 전체를 함께 평가합니다.
    .on_output()  # 최종 답변을 평가 대상에 넣습니다.
)


f_answer_relevance = (  # 답변이 질문과 관련 있는지 평가합니다.
    Metric(
        implementation=trulens_provider.relevance_with_cot_reasons,
        name="Answer Relevance",  # TruLens 화면에 표시될 지표 이름입니다.
    )
    .on_input()  # 사용자 질문을 평가 입력으로 사용합니다.
    .on_output()  # 최종 답변을 평가 입력으로 사용합니다.
)


f_context_relevance = (  # 검색 문맥이 질문과 관련 있는지 평가합니다.
    Metric(
        implementation=trulens_provider.context_relevance_with_cot_reasons,
        name="Context Relevance",  # TruLens 화면에 표시될 지표 이름입니다.
    )
    .on_input()  # 사용자 질문을 평가 입력으로 사용합니다.
    .on_context(collect_list=False)  # 검색 청크를 하나씩 평가합니다.
    .aggregate(np.mean)  # 청크별 평가 점수를 평균냅니다.
)


tru_recorder = TruChain(  # LangChain 체인을 TruLens 기록 대상으로 등록합니다.
    trulens_rag_chain,  # 위에서 만든 RAG 체인입니다.

    app_name="day3-rag-demo",  # TruLens에서 앱을 구분하는 이름입니다.
    app_version="retriever-k-2",  # 같은 앱 안에서 실험 버전을 구분합니다.

    feedbacks=[  # RAG 실행 후 계산할 평가 지표 목록입니다.
        f_groundedness,  # 답변이 검색 문맥에 근거했는지 평가합니다.
        f_answer_relevance,  # 답변이 질문과 관련 있는지 평가합니다.
        f_context_relevance,  # 검색 문맥이 질문과 관련 있는지 평가합니다.
    ],
)


print("TruLens recorder 준비 완료")  # TruLens 기록 객체 생성 완료 메시지입니다.


In [ ]:
trulens_question = "보이스피싱 피해금 환급절차에서 공고 기간은 얼마인가요?"  # 평가할 질문입니다.

with tru_recorder as recording:  # 이 블록 안의 RAG 실행을 TruLens가 기록합니다.
    trulens_answer = trulens_rag_chain.invoke(trulens_question)  # RAG 체인으로 답변을 생성합니다.

feedback_results = recording.retrieve_feedback_results(
    timeout=180  # feedback 계산이 끝날 때까지 최대 180초 기다립니다.
)

trulens_chunk_k = trulens_retriever.search_kwargs.get(
    "k", "확인 필요"  # Context Relevance 평균에 사용된 검색 청크 수입니다.
)

print("=== RAG 답변 ===")
print(trulens_answer)

print("\n검색된 청크 수:", trulens_chunk_k)  # Context Relevance는 청크별 점수의 평균입니다.
print("\n=== TruLens feedback 결과 ===")
feedback_results  # Groundedness/Answer Relevance/Context Relevance 점수를 확인합니다.


### 3.4 Langfuse, RAGAS, TruLens를 같이 볼 때

실제 운영에서는 도구 하나로 모든 문제를 해결하기보다, 역할을 나눠 보는 편이 낫습니다.

먼저 RAGAS와 TruLens는 평가를 붙이는 위치가 다릅니다.

- **RAGAS**는 평가용 샘플셋을 만들어 여러 케이스를 한 번에 비교·평가하는 방식에 강합니다.
- **TruLens**는 실제 RAG 체인을 실행할 때마다 record를 남기고, 그 실행 결과에 LLM 기반 feedback 평가를 붙이는 방식에 강합니다.

다르게 말하면, RAGAS는 `질문 / 검색 문맥 / 생성 답변 / 기준 답변`을 모아 둔 평가 데이터셋으로 batch 평가를 돌리는 도구에 가깝습니다.  
TruLens는 사용자가 실제로 호출한 RAG 체인의 입력, 검색 문맥, 출력 흐름을 기록하고, 그 record마다 `Groundedness`, `Answer Relevance`, `Context Relevance` 같은 feedback을 붙여 관찰하는 도구에 가깝습니다.

| 관점 | 주로 보는 질문 | 잘 맞는 도구 |
|---|---|---|
| 평가셋 기반 성능 비교 | 여러 질문에서 지금 방식이 이전보다 좋아졌는가 | RAGAS |
| 실행 흐름과 trace | 특정 요청이 어떤 단계에서 실패했는가 | Langfuse |
| 실행 record별 feedback | 이 요청의 응답이 근거 있는가, 질문과 관련 있는가 | TruLens |
| 운영 중 디버깅 | 특정 요청에서 어떤 문서가 검색되고 프롬프트에 들어갔는가 | Langfuse |

정리하면,

- **RAGAS**는 평가셋 중심의 batch 평가
- **Langfuse**는 운영 trace 중심의 실행 관찰
- **TruLens**는 실제 체인 실행 record와 feedback 중심의 품질 관찰

으로 이해하면 출발이 쉽습니다.


## 4. 청크 기반 RAG의 한계와 Graph RAG

### 4.1 청크 기반 RAG가 어려워하는 질문

기본 RAG는 문서를 청크로 나누고, 질문과 비슷한 청크를 찾아 답변에 넣습니다.
이 방식은 질문에 답이 들어 있는 문단을 바로 찾을 수 있을 때 잘 작동합니다.

하지만 답이 **여러 대상의 관계**를 따라가야 나오는 경우에는 흔들릴 수 있습니다.  
청크 검색은 기본적으로 "비슷한 문장 덩어리"를 찾는 방식이지, 회의실과 예약 절차, 준비물 사이의 연결을 따로 저장하고 따라가는 방식은 아니기 때문입니다.

예를 들어 사내 운영 문서가 아래처럼 나뉘어 있다고 해봅니다.

```text
[청크 1] 회의실 A는 총무팀에서 예약과 기본 비품 관리를 맡고 있습니다.
[청크 2] 회의실 사용 안내서에는 예약 절차와 장비 반납 절차가 포함되어 있습니다.
[청크 3] 예약 절차를 진행할 때는 사원증이 필요합니다.
```

`회의실 A를 예약하려면 어디에 연락하고 무엇을 준비해야 하나요?`라는 질문은 한 문단만 찾으면 끝나지 않습니다.  
이 질문에 답하려면 다음 정보를 함께 연결해야 합니다.  

- 회의실 A의 예약과 비품 관리는 총무팀이 맡고 있습니다.
- 회의실 사용 안내서에는 예약 절차가 설명되어 있습니다.
- 예약 절차를 진행할 때는 사원증이 필요합니다.

즉, 최종 답은 `총무팀에 연락하고 사원증을 준비해야 합니다.`가 됩니다.

<img src="image/graph_rag_query_flow.svg" width="900">

청크 기반 검색만 쓰면 이런 문제가 생길 수 있습니다.

| 한계 | 왜 생기는가 | 예시 |
|---|---|---|
| 필요한 정보가 여러 청크에 흩어짐 | 한 청크 안에 모든 관계가 들어 있지 않음 | 연락할 곳은 청크 1, 준비물은 청크 3에 있음 |
| 관계 방향이 흐려짐 | 비슷한 단어는 찾지만 `무엇이 무엇을 필요로 하는지`는 따로 저장하지 않음 | `예약 절차`가 `사원증`을 필요로 하는지, 반대로 사원증 설명인지 구분 필요 |
| 표현이 달라지면 연결이 끊김 | 문서마다 `담당 부서`, `관리 부서`, `예약 담당`처럼 표현이 다름 | 같은 관계를 다른 단어로 적음 |
| 다단계 질문에 약함 | 검색 결과를 많이 넣어도 중간 연결을 명시적으로 따라가지 않음 | 회의실 A에서 시작해 예약 절차와 준비물까지 이어서 찾아야 함 |


### 4.2 Graph RAG는 무엇을 보완하나

Graph RAG는 청크 기반 RAG의 한계를 어느 정도 보완합니다.
문서를 청크 덩어리로만 보지 않고, 중요한 대상을 **엔터티**로 잡고 두 대상의 연결을 **관계**로 저장합니다.

<img src="image/graph_rag_pipeline.svg" width="900">

여기서 핵심은 벡터DB가 아니라 **지식그래프 조회**입니다.
벡터 검색은 필요한 문맥을 보강할 때 함께 붙일 수 있습니다.

벡터 검색은 질문과 **비슷한 문장**을 찾는 데 강합니다.
그래프는 여러 대상이 **어떤 관계로 이어지는지**를 따라가는 데 강합니다.

Graph RAG가 특히 도움이 되는 상황은 아래처럼 정리할 수 있습니다.

| 상황 | 왜 그래프가 유리한가 |
|---|---|
| 단일 관계를 묻는 질문 | 회의실 A 예약은 총무팀에 연락해야 한다는 식으로 답이 특정 연결 정보에 묶여 있을 때 찾기 쉽습니다. |
| 여러 단계를 따라가야 하는 질문 | 회의실 A에서 시작해 예약 절차, 사전 승인, 필요 서류를 차례로 확인해야 하는 질문에 강합니다. |
| 여러 문서에 정보가 흩어진 경우 | 회의실 안내서, 예약 규정, 비품 관리 문서에 나뉜 정보를 `회의실 A` 같은 엔터티 기준으로 연결할 수 있습니다. |
| 표현이 조금씩 다른 경우 | `담당 부서`, `관리 부서`, `운영 부서`를 같은 관계명으로 정리해 검색 흔들림을 줄일 수 있습니다. |
| 답변 근거를 설명해야 하는 경우 | 왜 총무팀에 연락해야 하는지, 어떤 정보에서 답이 나왔는지 함께 보여주기 쉽습니다. |

따라서 Graph RAG는 청크 기반 RAG를 무조건 대체하는 기술이라기보다, **청크 검색만으로는 관계를 안정적으로 따라가기 어려운 질문을 보완하는 방법**으로 이해하면 좋습니다.
그래프를 바로 구현하기 전에, 먼저 온톨로지, 트리플, 지식그래프가 어떤 역할을 하는지 잡아둡니다.


### 4.3 지식그래프 기본 개념

지식그래프는 정보를 문장 덩어리로만 보지 않고, **대상과 관계**로 나누어 정리합니다.
이 구조를 쓰면 `어떤 공간이`, `어떤 물건을`, `어떤 절차로`, `어느 팀과 연결되는지`를 더 명확하게 다룰 수 있습니다.

먼저 아래 개념을 분리해 봅니다.


| 개념 | 뜻 | 간단한 예시 |
|---|---|---|
| 엔터티(Entity) | 그래프에서 하나의 노드가 되는 구별 가능한 대상 | `회의실 A`, `빔프로젝터`, `회의실 사용 안내서`, `총무팀` |
| 관계(Relation) | 두 엔터티를 방향 있게 연결하는 의미 | `갖추고있다`, `담당부서이다`, `포함한다`, `필요로한다` |
| 트리플(Triple) | 하나의 사실을 `주어-관계-목적어`로 표현한 단위 | `회의실 A - 갖추고있다 - 빔프로젝터` |
| 온톨로지(Ontology) | 지식그래프에서 사용할 관계 이름과 연결 규칙을 정한 설계도 | 관계: `갖추고있다`, `포함한다` |
| 지식그래프(KG) | 온톨로지 규칙에 따라 실제 엔터티와 관계를 연결한 데이터 그래프 | 관계 탐색과 추론에 사용 |

관계는 가능하면 명사 하나보다 **두 엔터티를 어떻게 연결하는지 드러나는 표현**으로 잡는 것이 좋습니다.  
`담당부서`, `준비물`처럼 속성명에 가까운 표현도 쓸 수 있지만, 처음에는 `담당부서이다`, `필요로한다`처럼 문장으로 읽히게 만들면 구분이 쉽습니다.

아래 그림처럼 대상은 노드가 되고, 관계는 방향이 있는 연결선이 됩니다.

<img src="image/knowledge_graph_example.svg" width="820">

온톨로지는 "그래프를 어떤 규칙으로 만들 것인가"를 정하고, 지식그래프는 그 규칙에 따라 쌓인 실제 데이터라고 보면 됩니다.


In [ ]:
ontology = {  # 예제에서 사용할 관계 이름만 정합니다.
    'relations': {'갖추고있다', '담당부서이다', '포함한다', '필요로한다'}
}

triples = [  # 하나의 사실을 주어-관계-목적어로 적습니다.
    ('회의실 A', '갖추고있다', '빔프로젝터'),
    ('회의실 A', '담당부서이다', '총무팀'),
    ('회의실 사용 안내서', '포함한다', '예약 절차'),
    ('예약 절차', '필요로한다', '사원증'),
]

print('허용 관계:', sorted(ontology['relations']))
print('예시 트리플:', triples[0])


이 예제에서는 타입 검증까지 다루지 않습니다.
먼저 **관계 이름을 정하고, 문장을 `주어-관계-목적어`로 바꾸는 흐름**만 보겠습니다.

그래프를 만들기 전에 **어떤 관계를 허용할지 먼저 정해야** 데이터가 뒤섞이지 않습니다.

예를 들어 `갖추고있다`, `포함한다`, `필요로한다`를 모두 같은 "관련 있음"으로 뭉개면,
그래프 탐색 단계에서 질문 의도에 맞는 관계를 고르기 어려워집니다.


In [ ]:
from collections import defaultdict  # 관계별 연결 목록을 만들 때 씁니다.

graph = defaultdict(list)  # 주어를 기준으로 관계와 목적어를 묶어 저장합니다.
for subject, relation, obj in triples:
    graph[subject].append((relation, obj))  # 같은 주어 아래에 관계-목적어를 계속 붙입니다.

def query_graph(subject, relation=None):  # 주어와 관계로 그래프를 조회합니다.
    records = graph.get(subject, [])  # 해당 주어에 연결된 모든 관계를 가져옵니다.
    if relation is None:
        return records  # 관계를 지정하지 않으면 전체를 돌려줍니다.
    return [obj for rel, obj in records if rel == relation]  # 관계가 맞는 목적어만 남깁니다.

print(query_graph('회의실 A'))  # 한 주어에 연결된 전체 정보를 봅니다.
print(query_graph('회의실 A', '담당부서이다'))  # 특정 관계만 골라 조회합니다.


## 5. 미니 Graph RAG 예제

### 5.1 LLM으로 트리플 추출하기

업무 문서는 보통 표준 문장으로 깔끔하게 쓰이지 않습니다.
여기서는 LLM을 사용해 자유롭게 적힌 메모에서 엔터티와 관계를 JSON 트리플로 추출합니다.

핵심은 LLM에게 관계명을 자유롭게 만들게 하지 않고, 앞에서 정한 관계 이름 안에서 추출하게 하는 것입니다.


아래 코드는 LLM이 읽을 비정형 업무 메모를 준비합니다. 트리플처럼 정돈하지 않고, 필요한 사실과 배경 문장을 함께 섞어 둡니다.

In [ ]:
sample_text = """
지난주부터 회의실 예약 관련 문의가 계속 들어와서 내용을 한 번 정리해 둡니다.
본관 3층에 있는 큰 회의실은 내부에서 그냥 회의실 A라고 부릅니다.
발표가 있는 회의는 보통 이 방을 쓰면 되고, 안에는 빔프로젝터와 화이트보드가 준비되어 있습니다.
화이트보드 마커는 자주 없어지니 사용 전에 한 번 확인해 주세요.
회의실 A는 총무팀에서 예약과 기본 비품 관리를 맡고 있습니다.
회의실 사용 안내서를 보면 예약 절차와 장비 반납 절차가 같이 설명되어 있습니다.
예약할 때는 사원증이 필요하고, 장비 반납 절차에서는 대여 확인서를 챙겨야 합니다.
참고로 점심시간 직후에는 예약이 많이 몰리니 일정 조율 메일은 전날 보내는 편이 좋습니다.
"""  # 구조가 정해져 있지 않은 업무 메모입니다.

print(sample_text.strip())  # LLM에 넣을 원문을 먼저 확인합니다.


아래 코드는 앞에서 정의한 관계 이름을 프롬프트에 넣고, `gpt-5-mini`가 JSON 트리플을 반환하도록 요청합니다.

In [ ]:
import json  # 온톨로지를 프롬프트에 넣기 쉬운 문자열로 바꿉니다.

from langchain_core.output_parsers import JsonOutputParser  # JSON 응답을 파이썬 객체로 바꿉니다.
from langchain_core.prompts import PromptTemplate  # 텍스트를 프롬프트에 넣기 위한 틀입니다.
from langchain_openai import ChatOpenAI  # 트리플 추출에 사용할 LLM입니다.

triple_prompt = PromptTemplate.from_template("""
다음 텍스트에서 지식그래프 트리플을 추출하세요.

규칙:
- relation은 반드시 온톨로지의 relations 안에 있는 값만 사용합니다.
- 텍스트에 명확히 있는 사실만 추출합니다.
- subject는 사용자가 질문할 핵심 대상 이름으로 정리합니다.
- 절차명은 텍스트에 나온 전체 이름을 유지합니다. 예: "예약"이 아니라 "예약 절차".
- 어떤 대상의 예약, 관리, 비품처럼 세부 행위가 붙어 있어도 담당 부서를 묻는 관계라면 subject는 그 대상 이름으로 둡니다.
- 위치, 시간, 일반 주의사항처럼 관계로 묶기 어려운 내용은 제외합니다.
- 출력은 JSON 배열만 작성합니다.
- 각 항목은 subject, relation, object 키를 가집니다.

온톨로지:
{ontology_json}

출력 예시:
[
  {{
    "subject": "회의실 A",
    "relation": "갖추고있다",
    "object": "빔프로젝터"
  }},
  {{
    "subject": "예약 절차",
    "relation": "필요로한다",
    "object": "사원증"
  }}
]

텍스트:
{text}
""")

triple_llm = ChatOpenAI(model="gpt-5-mini", temperature=0)  # 같은 입력에서 안정적인 추출 결과를 얻습니다.
triple_chain = triple_prompt | triple_llm | JsonOutputParser()  # 프롬프트, 모델, JSON 파서를 연결합니다.
ontology_for_prompt = {key: sorted(values) for key, values in ontology.items()}  # set을 보기 쉬운 목록으로 바꿉니다.
ontology_json = json.dumps(ontology_for_prompt, ensure_ascii=False, indent=2)  # LLM이 읽을 온톨로지 문자열입니다.

raw_triples = triple_chain.invoke({"text": sample_text, "ontology_json": ontology_json})  # LLM이 트리플을 만듭니다.
entity_aliases = {"예약": "예약 절차"}  # LLM이 줄여 쓴 엔터티 이름을 예제 기준으로 맞춥니다.

extracted_triples = []  # 정규화한 트리플만 담습니다.
seen_triples = set()  # 같은 트리플이 반복될 때 한 번만 남깁니다.
for item in raw_triples:
    subject = entity_aliases.get(item.get("subject"), item.get("subject"))
    relation = item.get("relation")
    obj = entity_aliases.get(item.get("object"), item.get("object"))
    key = (subject, relation, obj)
    if relation in ontology["relations"] and key not in seen_triples:
        extracted_triples.append({"subject": subject, "relation": relation, "object": obj})
        seen_triples.add(key)

extracted_triples


아래 코드는 LLM이 만든 트리플을 관계를 따라 조회하기 쉬운 작은 그래프 구조로 바꿉니다.

In [ ]:
mini_graph = defaultdict(list)  # 추출한 트리플을 조회용 그래프로 묶습니다.
for item in extracted_triples:
    subject = item["subject"]  # 주어 엔터티입니다.
    relation = item["relation"]  # 온톨로지에 넣어 둔 관계명입니다.
    obj = item["object"]  # 목적어 엔터티입니다.
    mini_graph[subject].append((relation, obj))  # 그래프 조회용으로 관계와 목적어를 저장합니다.

for subject, records in mini_graph.items():
    print(subject)
    for relation, obj in records:
        print(f"  - {relation}: {obj}")


LLM은 다양한 문장 표현에서 트리플 후보를 뽑는 데 유리합니다.
하지만 관계명을 마음대로 늘리면 그래프가 금방 지저분해집니다.

그래서 먼저 사용할 관계 이름을 정하고, LLM이 그 안에서 트리플을 만들게 합니다.
실무에서는 필요에 따라 검수와 정규화를 추가할 수 있지만, 여기서는 흐름을 보기 위해 최소 코드로 둡니다.

관계 이름이 안정적이면 `담당 부서`, `포함 절차`, `필요한 물건`처럼 질문 의도가 달라졌을 때도 그래프를 따라 필요한 값을 찾기 쉬워집니다.


### 5.2 그래프 조회 계획 만들기

트리플을 뽑아 작은 그래프를 만들었다면, 다음 단계는 질문을 그래프 조회 계획으로 바꾸는 것입니다.
단순히 질문에 특정 키워드가 들어 있는지 보는 방식은 실제 Graph RAG라고 보기 어렵습니다.
조금 더 정석적인 흐름은 **질문에서 엔터티와 관계 의도를 뽑고, 그래프의 노드와 관계에 매핑한 뒤, 그 관계를 따라 조회**하는 것입니다.

아래 예시는 운영용 그래프 DB 대신 앞에서 만든 `mini_graph`를 사용합니다.
실무에서는 이 조회 단계를 Neo4j의 Cypher 쿼리, 그래프 DB의 관계 탐색 기능, 또는 Graph RAG 도구가 대신합니다.


In [ ]:
known_entities = sorted(mini_graph.keys())  # 그래프에서 조회를 시작할 수 있는 노드 목록입니다.
known_relations = sorted(ontology['relations'])  # 그래프에서 허용하는 관계 목록입니다.

graph_query_prompt = PromptTemplate.from_template("""
사용자의 질문을 보고 그래프 조회 계획을 JSON으로 작성하세요.

[사용 가능한 엔터티]
{known_entities}

[사용 가능한 관계]
{known_relations}

[판단 기준]
- 질문이 특정 엔터티의 특정 관계를 묻는다면 graph_needed를 true로 둡니다.
- subject는 사용 가능한 엔터티 중 하나로 고릅니다.
- relation은 사용 가능한 관계 중 하나로 고릅니다.
- 그래프만으로 답하기 어렵거나 일반 요약 질문이면 graph_needed를 false로 둡니다.
- 확실하지 않은 값은 null로 둡니다.

[출력 형식]
{{
  "graph_needed": true,
  "subject": "회의실 A",
  "relation": "담당부서이다",
  "reason": "회의실 A의 담당 부서라는 특정 관계를 묻고 있음"
}}

질문: {query}
""")

graph_query_chain = graph_query_prompt | triple_llm | JsonOutputParser()  # 질문을 그래프 조회 계획으로 바꿉니다.

def plan_graph_query(query):  # 질문에서 엔터티와 관계 의도를 추출합니다.
    return graph_query_chain.invoke(
        {
            'query': query,
            'known_entities': known_entities,
            'known_relations': known_relations,
        }
    )

def lookup_mini_graph(plan):  # 조회 계획을 실제 그래프 조회로 바꿉니다.
    if not plan.get('graph_needed'):
        return []  # 그래프 조회가 맞지 않는 질문은 빈 결과로 둡니다.

    subject = plan.get('subject')
    relation = plan.get('relation')
    records = mini_graph.get(subject, [])  # subject 노드에 연결된 관계들을 가져옵니다.

    if relation:
        return [(rel, obj) for rel, obj in records if rel == relation]
    return records

test_queries = [  # 그래프 조회 계획을 비교할 질문들입니다.
    '회의실 A 예약은 어디에 연락해야 하나요?',
    '회의실 사용 안내서는 어떤 절차를 포함하는가?',
    '예약 절차에 필요한 물건은 무엇인가?',
    '회의실 예약 방법을 요약해줘',
]

for query in test_queries:
    plan = plan_graph_query(query)  # 질문을 엔터티-관계 조회 계획으로 바꿉니다.
    graph_result = lookup_mini_graph(plan)  # 계획대로 작은 그래프를 조회합니다.
    print('\n[질문]', query)
    print('[그래프 조회 계획]', plan)
    print('[그래프 조회 결과]', graph_result if graph_result else '그래프 조회보다 벡터 검색이나 일반 요약이 더 적합')


### 5.3 그래프 결과로 답변 만들기

그래프 조회 결과는 그 자체로 최종 답변이라기보다, 답변 생성에 넣을 **구조화된 문맥**에 가깝습니다.
아래 예시는 그래프에서 찾은 관계를 짧은 문맥으로 바꾼 뒤, LLM이 그 문맥만 보고 답변하게 합니다.


In [ ]:
def format_graph_context(plan, graph_result):  # 그래프 조회 결과를 LLM이 읽기 쉬운 문맥으로 바꿉니다.
    subject = plan.get("subject") or "알 수 없음"  # 조회를 시작한 엔터티입니다.
    if not graph_result:
        return "그래프에서 직접 확인한 관계가 없습니다."

    lines = [f"질문 대상: {subject}"]
    for relation, obj in graph_result:
        lines.append(f"- {subject} -> {relation} -> {obj}")  # 관계 경로를 한 줄씩 적습니다.
    return "\n".join(lines)

graph_answer_prompt = PromptTemplate.from_template("""
아래 그래프 조회 결과만 사용해 질문에 답하세요.
그래프 조회 결과에 답이 없으면 "그래프만으로는 답하기 어렵습니다."라고 답하세요.

질문:
{query}

그래프 조회 결과:
{graph_context}

답변:
""")

graph_answer_chain = graph_answer_prompt | triple_llm | StrOutputParser()  # 그래프 문맥을 자연어 답변으로 바꿉니다.

query = "회의실 A 예약은 어디에 연락해야 하나요?"
plan = plan_graph_query(query)  # 질문을 그래프 조회 계획으로 바꿉니다.
graph_result = lookup_mini_graph(plan)  # 계획에 맞게 작은 그래프를 조회합니다.
graph_context = format_graph_context(plan, graph_result)  # 조회 결과를 답변 문맥으로 정리합니다.

print("[그래프 조회 문맥]")
print(graph_context)
print("\n[답변]")
if graph_result:
    answer = graph_answer_chain.invoke({"query": query, "graph_context": graph_context})
    print(answer)
else:
    print("그래프만으로는 답하기 어렵습니다.")


여기서는 작은 딕셔너리 그래프로 흐름만 확인했습니다.
실무에서는 그래프DB, Cypher 같은 그래프 질의어, 벡터 검색 fallback, 다단계 경로 탐색을 함께 설계합니다.


### 5.4 Graph RAG 설계 체크리스트

Graph RAG를 바로 구현하기 전에 아래를 먼저 정리하면 훨씬 수월합니다.

1. 문서에서 반복적으로 등장하는 핵심 엔터티가 무엇인가
2. 질문에서 자주 물어보는 관계가 무엇인가
3. 관계 이름을 어느 수준까지 세분화할 것인가
4. 청크 검색으로 충분한 질문과 그래프 조회가 필요한 질문을 어떻게 나눌 것인가
5. 그래프 결과를 다시 생성 단계에 어떻게 넘길 것인가

결국 Graph RAG는 청크 기반 RAG를 버리는 방향이 아니라, **관계 탐색이 중요한 질문을 더 잘 처리하기 위한 확장**으로 이해하면 좋습니다.

<img src="image/rag_evolution_graph.svg" width="820">


## 6. 미니 RAG 경진대회 안내

미니 RAG 경진대회는 별도 노트북 **`미니RAG경진대회.ipynb`** 에서 진행합니다.

여기서는 전체 흐름만 확인합니다.

1. `미니RAG경진대회.ipynb`의 baseline을 실행합니다.
2. Retrieval / Post-Retrieval / Generation 중 개선 포인트를 적용합니다.
3. RAGAS 점수와 실패 사례를 확인합니다.
4. 필요하면 Langfuse trace로 특정 요청의 검색된 청크와 답변 흐름을 점검합니다.
5. 최종 점수와 시도한 개선 방법을 제출 문서에 정리합니다.
